# Model 0: Optimised Naive Baseline Model for Drug `N05C`

## Hyperparameter Selection Methodology:
Grid search across candidate lags $k \in \{1, 2, 7, 14, 21, 28, 30, 60, 90, 365\}$ on the 2018 Validation Set to identify $k^*$.


In [1]:
# Dynamic Dependency Guard & Environment Initialization
import sys, subprocess, os

def install_and_import(pkg, module_name=None):
    if module_name is None:
        module_name = pkg
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import('numpy')
install_and_import('pandas')
install_and_import('matplotlib')
install_and_import('seaborn')
install_and_import('scikit-learn', 'sklearn')
install_and_import('statsmodels')
install_and_import('lightgbm')
install_and_import('xgboost')
install_and_import('shap')
install_and_import('prophet')
install_and_import('optuna')
install_and_import('torch')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.sans-serif': 'Inter, Roboto, Arial, sans-serif',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--'
})

TARGET_DRUG = 'N05C'
data_dir = r'c:\Users\ranje\sales forcasting\times_series\dataset'

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df   = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series   = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series  = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series     = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    rmse  = np.sqrt(np.mean((y_true - y_pred)**2))
    mae   = np.mean(np.abs(y_true - y_pred))
    wape  = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE (%)': wape}

print(f"Dataset for {TARGET_DRUG} loaded successfully!")
print(f"  * Train  : {train_series.index.min().strftime('%Y-%m-%d')} to {train_series.index.max().strftime('%Y-%m-%d')} ({len(train_series)} days)")
print(f"  * Val    : {val_series.index.min().strftime('%Y-%m-%d')} to {val_series.index.max().strftime('%Y-%m-%d')} ({len(val_series)} days)")
print(f"  * Test   : {test_series.index.min().strftime('%Y-%m-%d')} to {test_series.index.max().strftime('%Y-%m-%d')} ({len(test_series)} days)")


Dataset for N05C loaded successfully!
  * Train  : 2014-01-02 to 2017-12-31 (1460 days)
  * Val    : 2018-01-01 to 2018-12-31 (365 days)
  * Test   : 2019-01-01 to 2019-10-08 (281 days)


In [2]:
# Step 1: Lag Grid Search Code Execution on 2018 Validation Set
candidate_lags = [1, 2, 7, 14, 21, 28, 30, 60, 90, 365]
val_results = {}

for k in candidate_lags:
    pred_val = train_series.values[-k:][:len(val_series)] if k >= len(val_series) else combined_series.shift(k).loc[val_series.index]
    val_results[k] = evaluate_metrics(val_series.values, pred_val)['RMSLE']

best_lag = min(val_results, key=val_results.get)
print("=== 2018 Validation Set Lag Grid Search Results ===")
for k, rmsle in val_results.items():
    star = " <--- OPTIMAL (k*)" if k == best_lag else ""
    print(f"  * Lag k = {k:3d} days : Val RMSLE = {rmsle:.6f}{star}")

print(f"Selected Optimal Seasonal Lag parameter: k* = {best_lag} days")


=== 2018 Validation Set Lag Grid Search Results ===
  * Lag k =   1 days : Val RMSLE = 0.745088
  * Lag k =   2 days : Val RMSLE = 0.753298
  * Lag k =   7 days : Val RMSLE = 0.719185
  * Lag k =  14 days : Val RMSLE = 0.745554
  * Lag k =  21 days : Val RMSLE = 0.784545
  * Lag k =  28 days : Val RMSLE = 0.716931
  * Lag k =  30 days : Val RMSLE = 0.723332
  * Lag k =  60 days : Val RMSLE = 0.711093 <--- OPTIMAL (k*)
  * Lag k =  90 days : Val RMSLE = 0.750903
  * Lag k = 365 days : Val RMSLE = 0.712954
Selected Optimal Seasonal Lag parameter: k* = 60 days


In [3]:
# Step 2: Forecast 2019 Test Holdout using Selected Optimal Lag k*
m0_test_pred = full_series.shift(best_lag).loc[test_series.index].fillna(combined_series.mean())
test_metrics = evaluate_metrics(test_series, m0_test_pred)

print(f"=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 0: OPTIMISED NAIVE (k* = {best_lag}) ===")
for k, v in test_metrics.items():
    print(f"  * {k:10s}: {v:.4f}")

pd.DataFrame({'date': test_series.index, 'pred_Naive': m0_test_pred.values}).to_csv('m0_naive_preds.csv', index=False)


=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 0: OPTIMISED NAIVE (k* = 60) ===
  * RMSLE     : 0.7222
  * RMSE      : 1.5441
  * MAE       : 1.0249
  * WAPE (%)  : 146.9388
